# LeetCode 3797 — Count Routes to Climb a Rectangular Grid bằng Python

- Link: https://leetcode.com/problems/count-routes-to-climb-a-rectangular-grid/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động theo hàng với hai trạng thái, tổng tiền tố cho tổng cửa sổ, khoảng cách Euclid → khoảng cột cho phép, modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfRoutes(self, grid: List[str], d: int) -> int:
```

## 1. Mục tiêu học

1. Đổi ràng buộc **khoảng cách Euclid ≤ d** thành **khoảng cột**: đi ngang `|Δc| ≤ d`; đi lên một hàng `|Δc| ≤ ⌊√(d² − 1)⌋`.
2. Thiết kế DP **hai trạng thái** mỗi ô: vừa **đi lên** tới (được phép đi ngang tiếp) / vừa **đi ngang** tới (bắt buộc đi lên, hoặc kết thúc).
3. Tính tổng trên **cửa sổ cột** bằng tổng tiền tố ⇒ mỗi hàng `O(m)`.
4. Xử lý đúng: điểm bắt đầu ở hàng dưới cùng, kết thúc ở hàng trên cùng, ô bị chặn, và nước đi ngang phải tới **ô khác**.
5. Kiểm thử với DFS liệt kê mọi tuyến đường.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Lưới `n × m`, ô `'.'` mở, `'#'` chặn. Đếm số tuyến đường:

- bắt đầu ở một ô mở của **hàng dưới cùng** (`n − 1`), kết thúc ở **hàng trên cùng** (`0`);
- mỗi nước đi tới một ô mở **khác**, khoảng cách Euclid `≤ d`;
- mỗi nước đi hoặc **giữ nguyên hàng**, hoặc lên **đúng một** hàng;
- **không** được giữ nguyên hàng hai nước liên tiếp.

Trả về số tuyến modulo `10^9 + 7`.

**Ví dụ 1:** `grid = ["..","#."], d = 1` → `2`. **Ví dụ 2:** cùng lưới, `d = 2` → `4`.

**Ví dụ 3:** `grid = ["#"], d = 750` → `0`. **Ví dụ 4:** `grid = [".."], d = 1` → `4` (hai tuyến một ô, hai tuyến đi ngang).

**Ràng buộc:** `1 ≤ n, m ≤ 750`, `1 ≤ d ≤ 750`.

In [1]:
from typing import List
from math import isqrt
import random, time

MOD = 10**9 + 7
examples = [(["..", "#."], 1, 2), (["..", "#."], 2, 4), (["#"], 750, 0), ([".."], 1, 4)]
for g, d, exp in examples:
    print(g, "d =", d, "-> kỳ vọng", exp)

['..', '#.'] d = 1 -> kỳ vọng 2
['..', '#.'] d = 2 -> kỳ vọng 4
['#'] d = 750 -> kỳ vọng 0
['..'] d = 1 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Khoảng cách Euclid → khoảng cột

- Cùng hàng: `√(Δc²) = |Δc| ≤ d`.
- Lên một hàng: `√(1 + Δc²) ≤ d` ⇔ `Δc² ≤ d² − 1` ⇔ `|Δc| ≤ ⌊√(d² − 1)⌋` (dùng `math.isqrt` để tránh sai số số thực).

In [2]:
for d in (1, 2, 3, 5, 750):
    print(f"d={d}: ngang |Δc| ≤ {d}, lên |Δc| ≤ {isqrt(d * d - 1)}")

d=1: ngang |Δc| ≤ 1, lên |Δc| ≤ 0
d=2: ngang |Δc| ≤ 2, lên |Δc| ≤ 1
d=3: ngang |Δc| ≤ 3, lên |Δc| ≤ 2
d=5: ngang |Δc| ≤ 5, lên |Δc| ≤ 4
d=750: ngang |Δc| ≤ 750, lên |Δc| ≤ 749


### 3.2. Tổng cửa sổ bằng tổng tiền tố

`pre[j] = a[0] + … + a[j−1]`. Tổng `a[lo..hi]` = `pre[hi + 1] − pre[lo]`. Với mỗi cột `c`, cửa sổ `[c − w, c + w]` cắt vào `[0, m − 1]`.

In [3]:
from itertools import accumulate
a = [3, 0, 2, 5, 1]
pre = list(accumulate(a, initial=0))
w = 1
print([pre[min(len(a), c + w + 1)] - pre[max(0, c - w)] for c in range(len(a))])

[3, 5, 7, 8, 6]


### 3.3. Ràng buộc "không đi ngang hai lần liên tiếp" ⇒ hai trạng thái

Gọi trạng thái tại ô:

- `A` — vừa **tới bằng nước đi lên** (hoặc là ô bắt đầu): được đi ngang **hoặc** đi lên.
- `B` — vừa **tới bằng nước đi ngang**: chỉ được đi lên (hoặc kết thúc nếu đang ở hàng 0).

Đây là kỹ thuật thêm "bộ nhớ ngắn hạn" vào trạng thái DP.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi hàng được "ghé" theo một trong hai kiểu: chỉ một ô (tới rồi đi lên ngay), hoặc hai ô (tới, đi ngang một lần, rồi đi lên). Vì không bao giờ quay xuống, ta xử lý **từ hàng dưới lên**.

Tính tay ví dụ 2 (`d = 2` ⇒ ngang `|Δc| ≤ 2`, lên `|Δc| ≤ 1`):

- Hàng 1 (`#.`): bắt đầu ở `(1,1)`: `A[1] = [0, 1]`; đi ngang sang `(1,0)` bị chặn ⇒ `B[1] = [0, 0]`.
- Lên hàng 0: mỗi ô `(0, c)` nhận tổng `A + B` của hàng 1 trong cửa sổ `|Δc| ≤ 1`: `A[0] = [1, 1]`.
- Đi ngang trong hàng 0: `B[0][0] = A[0][1] = 1`, `B[0][1] = A[0][0] = 1`.
- Kết thúc ở hàng 0 (cả hai trạng thái): `1 + 1 + 1 + 1 = 4` ✔.

## 5. Cách 1 — Brute Force: DFS liệt kê mọi tuyến

Từ mỗi ô mở ở hàng dưới, DFS với trạng thái "nước trước có phải đi ngang không". Mỗi khi đang ở hàng 0 thì đếm một tuyến (tuyến có thể kết thúc ở đó). Hàm mũ — chỉ cho lưới nhỏ.

In [4]:
def brute(grid, d):
    n, m = len(grid), len(grid[0])
    def dfs(r, c, last_horizontal):
        total = 1 if r == 0 else 0                   # kết thúc tại đây (nếu ở hàng trên cùng)
        for nr in (r, r - 1):
            if nr < 0: continue
            if nr == r and last_horizontal: continue
            for nc in range(m):
                if grid[nr][nc] != '.' or (nr == r and nc == c): continue
                if (r - nr) ** 2 + (c - nc) ** 2 <= d * d:
                    total += dfs(nr, nc, nr == r)
        return total
    return sum(dfs(n - 1, c, False) for c in range(m) if grid[n - 1][c] == '.') % MOD

for g, d, exp in examples:
    print(brute(g, d), "kỳ vọng", exp)

2 kỳ vọng 2
4 kỳ vọng 4
0 kỳ vọng 0
4 kỳ vọng 4


**Nút thắt:** số tuyến tăng theo cấp số nhân với số hàng. Nhưng số tuyến tới một ô với một trạng thái chỉ phụ thuộc **(hàng, cột, trạng thái)** ⇒ DP.

## 6. Ý tưởng thuật toán / bước đột phá

**Định nghĩa** (chỉ với ô mở):

- `A[r][c]` = số cách tới `(r, c)` mà nước cuối là **đi lên** (hoặc đây là điểm bắt đầu khi `r = n − 1`);
- `B[r][c]` = số cách tới `(r, c)` mà nước cuối là **đi ngang**.

**Chuyển trạng thái:**

1. Khởi tạo `A[n−1][c] = 1` với ô mở.
2. Trong hàng `r`: `B[r][c] = Σ_{c' ≠ c, |c − c'| ≤ d} A[r][c']`.
3. Lên hàng `r − 1`: `A[r−1][c] = Σ_{|c − c'| ≤ d₁} (A[r][c'] + B[r][c'])` với `d₁ = ⌊√(d² − 1)⌋`.
4. Đáp án `Σ_c (A[0][c] + B[0][c])`.

Mỗi tổng là tổng cửa sổ ⇒ tổng tiền tố ⇒ `O(m)` mỗi hàng, tổng `O(n · m)`.

Lưu ý `c' ≠ c` ở bước 2: tổng cửa sổ trừ đi chính `A[r][c]`.

## 7. Các cách tiếp theo

- **Cách 2:** DP hai trạng thái với tổng cửa sổ tính **trực tiếp** — `O(n · m · d)`.
- **Cách 3:** DP hai trạng thái + **tổng tiền tố** — `O(n · m)` (lời giải nộp chính).

### 7.1. Cách 2 — DP với tổng cửa sổ trực tiếp

In [5]:
def dp_naive(grid, d):
    n, m = len(grid), len(grid[0])
    d1 = isqrt(d * d - 1)
    A = [1 if grid[n - 1][c] == '.' else 0 for c in range(m)]
    for r in range(n - 1, -1, -1):
        B = [0] * m
        for c in range(m):
            if grid[r][c] == '.':
                B[c] = sum(A[k] for k in range(max(0, c - d), min(m, c + d + 1)) if k != c) % MOD
        if r == 0:
            return (sum(A) + sum(B)) % MOD
        T = [(A[c] + B[c]) for c in range(m)]
        A = [0] * m
        for c in range(m):
            if grid[r - 1][c] == '.':
                A[c] = sum(T[k] for k in range(max(0, c - d1), min(m, c + d1 + 1))) % MOD

for g, d, exp in examples:
    print(dp_naive(g, d), "kỳ vọng", exp)

2 kỳ vọng 2
4 kỳ vọng 4
0 kỳ vọng 0
4 kỳ vọng 4


**Độ phức tạp:** `O(n · m · d)` — tới `750³ ≈ 4·10^8`, quá chậm.

### 7.2. Cách 3 — DP + tổng tiền tố (lời giải chính)

**Giải thích từng đoạn:**

1. `A` cho hàng dưới cùng.
2. Mỗi hàng: tiền tố của `A` ⇒ `B[c] = (tổng cửa sổ ngang) − A[c]` (với ô mở).
3. Nếu là hàng 0 ⇒ trả `Σ(A + B)`.
4. Tiền tố của `A + B` ⇒ `A` mới của hàng trên = tổng cửa sổ `d₁` (với ô mở).

In [6]:
def window_sums(arr, w):
    m = len(arr)
    pre = [0] * (m + 1)
    for i, x in enumerate(arr): pre[i + 1] = pre[i] + x
    return [pre[min(m, c + w + 1)] - pre[max(0, c - w)] for c in range(m)]

def dp_prefix(grid, d):
    n, m = len(grid), len(grid[0])
    d1 = isqrt(d * d - 1)
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        row = grid[r]
        W = window_sums(A, d)
        B = [(W[c] - A[c]) % MOD if row[c] == '.' else 0 for c in range(m)]
        if r == 0:
            return (sum(A) + sum(B)) % MOD
        T = [(A[c] + B[c]) % MOD for c in range(m)]
        U = window_sums(T, d1)
        up = grid[r - 1]
        A = [U[c] % MOD if up[c] == '.' else 0 for c in range(m)]

for g, d, exp in examples:
    print(dp_prefix(g, d), "kỳ vọng", exp)

2 kỳ vọng 2
4 kỳ vọng 4
0 kỳ vọng 0
4 kỳ vọng 4


**Độ phức tạp Cách 3:** mỗi hàng `O(m)` ⇒ `O(n · m)` ≈ `5,6·10^5` ô; bộ nhớ `O(m)`.

## 8. TRACE từng bước

Trace Cách 3 trên lưới `["...", ".#.", "..."]` với `d = 1` (lên chỉ được thẳng đứng `d₁ = 0`).

In [7]:
def trace(grid, d):
    n, m = len(grid), len(grid[0]); d1 = isqrt(d * d - 1)
    print(f"d={d}: ngang |Δc| ≤ {d}, lên |Δc| ≤ {d1}")
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d)
        B = [(W[c] - A[c]) if grid[r][c] == '.' else 0 for c in range(m)]
        print(f"hàng {r} '{grid[r]}': A={A} B={B}")
        if r == 0:
            print("đáp án =", sum(A) + sum(B)); return
        U = window_sums([a + b for a, b in zip(A, B)], d1)
        A = [U[c] if grid[r - 1][c] == '.' else 0 for c in range(m)]

trace(["...", ".#.", "..."], 1)
print("vét cạn:", brute(["...", ".#.", "..."], 1))

d=1: ngang |Δc| ≤ 1, lên |Δc| ≤ 0
hàng 2 '...': A=[1, 1, 1] B=[1, 2, 1]
hàng 1 '.#.': A=[2, 0, 2] B=[0, 0, 0]
hàng 0 '...': A=[2, 0, 2] B=[0, 4, 0]
đáp án = 8
vét cạn: 8


In [8]:
trace(["..", "#."], 2)

d=2: ngang |Δc| ≤ 2, lên |Δc| ≤ 1
hàng 1 '#.': A=[0, 1] B=[0, 0]
hàng 0 '..': A=[1, 1] B=[1, 1]
đáp án = 4


## 9. Code nộp LeetCode

In [9]:
from math import isqrt

class Solution:
    def numberOfRoutes(self, grid: List[str], d: int) -> int:
        MOD = 10**9 + 7
        n, m = len(grid), len(grid[0])
        d1 = isqrt(d * d - 1)                    # đi lên: 1 + dc^2 <= d^2

        def window(arr, w):
            pre = [0] * (m + 1)
            for i, x in enumerate(arr):
                pre[i + 1] = pre[i] + x
            return [pre[min(m, c + w + 1)] - pre[max(0, c - w)] for c in range(m)]

        A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
        for r in range(n - 1, -1, -1):
            row = grid[r]
            W = window(A, d)
            B = [(W[c] - A[c]) % MOD if row[c] == '.' else 0 for c in range(m)]
            if r == 0:
                return (sum(A) + sum(B)) % MOD
            U = window([(A[c] + B[c]) % MOD for c in range(m)], d1)
            up = grid[r - 1]
            A = [U[c] % MOD if up[c] == '.' else 0 for c in range(m)]
        return 0

sol = Solution()
for g, d, exp in examples:
    got = sol.numberOfRoutes(g, d)
    print(g, d, "->", got, "✔" if got == exp else "✘")

['..', '#.'] 1 -> 2 ✔
['..', '#.'] 2 -> 4 ✔
['#'] 750 -> 0 ✔
['..'] 1 -> 4 ✔


## 10. Vì sao thuật toán đúng

1. **Phân rã tuyến theo hàng:** vì không đi xuống và không đi ngang hai lần liên tiếp, phần tuyến trong một hàng là một ô hoặc hai ô (đi ngang một lần). Tuyến kết thúc ở hàng 0 (ngay khi tới bằng nước lên, hoặc sau một nước ngang).
2. **Trạng thái đủ thông tin:** để biết nước tiếp theo hợp lệ, chỉ cần vị trí và việc nước trước có phải đi ngang. `A`, `B` đếm đúng số tiền tố tuyến kết thúc tại ô đó với trạng thái tương ứng (quy nạp từ hàng dưới).
3. **Chuyển trạng thái:** đi ngang chỉ xuất phát từ trạng thái `A` (vì từ `B` cấm đi ngang), tới ô khác (`c' ≠ c`) trong `|Δc| ≤ d`; đi lên xuất phát từ cả `A` và `B`, `|Δc| ≤ d₁`. Ô bị chặn có giá trị 0.
4. **Kết thúc:** đáp án đếm mọi tuyến dừng ở hàng 0 ở cả hai trạng thái; tuyến không thể tiếp tục từ hàng 0 lên trên.
5. **`d₁`:** `1 + Δc² ≤ d²` ⇔ `|Δc| ≤ ⌊√(d² − 1)⌋` (số nguyên, tính bằng `isqrt` không sai số).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DFS | hàm mũ | `O(n)` ngăn xếp |
| 2. DP + cửa sổ trực tiếp | `O(n · m · d)` | `O(m)` |
| 3. DP + tổng tiền tố | `O(n · m)` | `O(m)` |

## 12. Bộ test

In [10]:
def run_all(grid, d, with_brute=True):
    res = [dp_naive(grid, d), dp_prefix(grid, d), Solution().numberOfRoutes(grid, d)]
    if with_brute:
        res.append(brute(grid, d))
    return res

for g, d, exp in examples:
    assert all(v == exp for v in run_all(g, d))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng DFS)
cases = [
    (["."], 1, 1),                    # một ô: một tuyến
    (["..."], 1, 3 + 4),              # 3 tuyến một ô + 4 nước ngang (|Δc| = 1)
    (["...", "###"], 5, 0),           # hàng dưới bị chặn hết
    ([".", ".", "."], 1, 1),          # cột đơn: chỉ đi thẳng lên
    (["#.", ".#"], 1, 0),             # d = 1 không đi chéo được
    (["#.", ".#"], 2, 1),             # d = 2 đi chéo được
]
for g, d, exp in cases:
    r = run_all(g, d)
    assert all(v == exp for v in r), (g, d, r)
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 phản ví dụ: dùng cùng bán kính d cho nước đi lên (quên +1 hàng trong khoảng cách)
def wrong_same_radius(grid, d):
    n, m = len(grid), len(grid[0])
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d); B = [(W[c] - A[c]) if grid[r][c] == '.' else 0 for c in range(m)]
        if r == 0: return (sum(A) + sum(B)) % MOD
        U = window_sums([a + b for a, b in zip(A, B)], d)          # SAI: phải là d1
        A = [U[c] if grid[r - 1][c] == '.' else 0 for c in range(m)]

print("sai:", wrong_same_radius(["..", "#."], 1), "| đúng:", Solution().numberOfRoutes(["..", "#."], 1))

sai: 4 | đúng: 2


In [13]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3797)
for _ in range(300):
    n, m = rng.randint(1, 4), rng.randint(1, 4)
    g = ["".join(rng.choice("..#") for _ in range(m)) for _ in range(n)]
    d = rng.randint(1, 4)
    r = run_all(g, d)
    assert len(set(r)) == 1, (g, d, r)
print("300 test ngẫu nhiên (có DFS): OK")

300 test ngẫu nhiên (có DFS): OK


In [14]:
# 12.5 test lớn đúng ràng buộc (750 x 750)
rng = random.Random(9)
g = ["".join("#" if rng.random() < 0.1 else "." for _ in range(750)) for _ in range(750)]
for d in (1, 30, 750):
    t0 = time.time(); a = Solution().numberOfRoutes(g, d); t1 = time.time()
    print(f"d={d}: {a}, thời gian {t1 - t0:.2f}s")
small = [row[:40] for row in g[:40]]
print("khớp DP trực tiếp trên lưới 40x40, d=7:", Solution().numberOfRoutes(small, 7) == dp_naive(small, 7))

d=1: 821459291, thời gian 0.34s


d=30: 347040149, thời gian 0.37s


d=750: 253298158, thời gian 0.32s
khớp DP trực tiếp trên lưới 40x40, d=7: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bán kính đi lên dùng `d` thay vì `⌊√(d² − 1)⌋`

Đã thấy ở 12.3: với `d = 1` chỉ được đi thẳng lên; dùng nhầm `d` cho phép đi chéo `(1,1) → (0,0)`.

### Lỗi 2 — Cho phép "đi ngang" tới chính ô đang đứng

Nếu không trừ `A[c]`, mỗi ô tự đóng góp vào `B` của chính nó — như một nước đi đứng yên.

In [15]:
def wrong_self(grid, d):
    n, m = len(grid), len(grid[0]); d1 = isqrt(d * d - 1)
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d); B = [W[c] if grid[r][c] == '.' else 0 for c in range(m)]     # SAI: chưa trừ A[c]
        if r == 0: return (sum(A) + sum(B)) % MOD
        U = window_sums([a + b for a, b in zip(A, B)], d1)
        A = [U[c] if grid[r - 1][c] == '.' else 0 for c in range(m)]

print("sai:", wrong_self(["."], 1), wrong_self([".."], 1), "| đúng:", Solution().numberOfRoutes(["."], 1), Solution().numberOfRoutes([".."], 1))

sai: 2 6 | đúng: 1 4


### Lỗi 3 — Cho phép đi ngang từ trạng thái `B` (hai lần liên tiếp)

In [16]:
def wrong_double_horizontal(grid, d):
    return brute_no_rule(grid, d)

def brute_no_rule(grid, d):
    # DP sai: B lấy từ (A + B) thay vì chỉ A -> cho phép nhiều nước ngang liên tiếp (giới hạn 2 lần để hữu hạn)
    n, m = len(grid), len(grid[0]); d1 = isqrt(d * d - 1)
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d); B = [(W[c] - A[c]) if grid[r][c] == '.' else 0 for c in range(m)]
        W2 = window_sums(B, d); B2 = [(W2[c] - B[c]) if grid[r][c] == '.' else 0 for c in range(m)]   # SAI: nước ngang thứ hai
        if r == 0: return (sum(A) + sum(B) + sum(B2)) % MOD
        U = window_sums([a + b + b2 for a, b, b2 in zip(A, B, B2)], d1)
        A = [U[c] if grid[r - 1][c] == '.' else 0 for c in range(m)]

print("sai:", wrong_double_horizontal(["..."], 2), "| đúng:", Solution().numberOfRoutes(["..."], 2))

sai: 21 | đúng: 9


### Lỗi 4 — Tính `d₁` bằng `int(math.sqrt(d*d − 1))` với số thực

Với `d` nhỏ thì không sao, nhưng thói quen tốt là dùng `isqrt` (chính xác tuyệt đối). Ở đây `d ≤ 750` nên `sqrt` thực cũng đúng — ô dưới kiểm tra.

In [17]:
import math
print("sqrt thực khớp isqrt với mọi d ≤ 750:", all(int(math.sqrt(d * d - 1)) == isqrt(d * d - 1) for d in range(1, 751)))
print("với d lớn (10^8):", int(math.sqrt((10**8) ** 2 - 1)), isqrt((10**8) ** 2 - 1))

sqrt thực khớp isqrt với mọi d ≤ 750: True
với d lớn (10^8): 100000000 99999999


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Không được làm X hai lần liên tiếp"** ⇒ thêm trạng thái "nước trước là X".
- **Ràng buộc khoảng cách** giữa các ô hàng kề nhau ⇒ đổi thành **khoảng cột** rồi dùng tổng cửa sổ.
- **DP theo hàng với chuyển trạng thái là tổng cửa sổ** ⇒ tổng tiền tố, mỗi hàng `O(m)`.
- Liên quan: LC 1594/1301 (DP lưới), LC 2435 (đếm đường trên lưới với modulo), LC 1937 (max điểm theo hàng với chi phí khoảng cách).

## 15. Biến thể của bài

1. **Được đi ngang tối đa `t` lần liên tiếp**: `t + 1` trạng thái.
2. **Khoảng cách Manhattan** thay cho Euclid: đi lên `|Δc| ≤ d − 1`.
3. **Có trọng số ô** và hỏi tổng trọng số lớn nhất: thay tổng cửa sổ bằng **max cửa sổ** (deque đơn điệu).
4. **Bắt đầu/kết thúc tùy ý**: khởi tạo `A` ở mọi hàng và cộng đáp án ở mọi hàng.

In [18]:
# Biến thể 2: khoảng cách Manhattan
def manhattan_routes(grid, d):
    n, m = len(grid), len(grid[0]); d1 = d - 1
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d); B = [(W[c] - A[c]) % MOD if grid[r][c] == '.' else 0 for c in range(m)]
        if r == 0: return (sum(A) + sum(B)) % MOD
        U = window_sums([(a + b) % MOD for a, b in zip(A, B)], d1)
        A = [U[c] % MOD if grid[r - 1][c] == '.' else 0 for c in range(m)]

print(manhattan_routes(["..", "#."], 2), "(Manhattan d=2: đi chéo tốn 2 -> được phép)", manhattan_routes(["..", "#."], 1))

4 (Manhattan d=2: đi chéo tốn 2 -> được phép) 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS | Liệt kê tuyến | hàm mũ | `O(n)` | Kiểm thử |
| 2. DP cửa sổ trực tiếp | Hai trạng thái, cộng từng ô | `O(nmd)` | `O(m)` | `d` nhỏ |
| 3. DP + tiền tố | Tổng cửa sổ `O(1)` | `O(nm)` | `O(m)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `numberOfRoutes([".", "."], 1)` = ?

**Bài 2 (tính tay).** `numberOfRoutes(["...", "..."], 1)`.

**Bài 3 (điền TODO).** Viết `allowed_up_radius(d)` và kiểm tra với vét cạn các cặp `(1, Δc)`.

**Bài 4 (tự viết lại).** Viết lại Cách 3 không tạo list `T` riêng (gộp tiền tố của `A + B` trong một vòng lặp).

**Bài 5 (mini challenge).** Biến thể "được đi ngang tối đa hai lần liên tiếp". Viết DP ba trạng thái và kiểm chứng bằng DFS.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def allowed_up_radius(d):
    # TODO: isqrt(d*d - 1)
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def two_horizontal(grid, d):
    # TODO: A (sau nước lên), B1 (một nước ngang), B2 (hai nước ngang liên tiếp)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cột đơn hai hàng. (2) Chỉ một tuyến: đi thẳng lên. (3) `1`.

**Bài 2.** (1) Hàng dưới `A = [1,1,1]`, `B = [1,2,1]`. (2) Lên thẳng: `A₀ = [2,3,2]`. (3) Hàng trên `B₀ = [3, 4, 3]` ⇒ tổng `7 + 10 = 17`.

**Bài 3.** (1) Duyệt `Δc` từ 0. (2) Điều kiện `1 + Δc² ≤ d²`. (3) So với `isqrt`.

**Bài 4.** (1) Tiền tố tính trên `A[c] + B[c]` trực tiếp. (2) Một mảng `pre`. (3) Đối chiếu.

**Bài 5.** (1) `B1` từ `A`, `B2` từ `B1` (ô khác). (2) Lên từ `A + B1 + B2`. (3) DFS với bộ đếm số nước ngang liên tiếp.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().numberOfRoutes([".", "."], 1) == 1
assert Solution().numberOfRoutes(["...", "..."], 1) == 17 == brute(["...", "..."], 1)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def allowed_up_radius_sol(d):
    return isqrt(d * d - 1)

for d in range(1, 60):
    assert allowed_up_radius_sol(d) == max(dc for dc in range(d + 1) if 1 + dc * dc <= d * d)

def dp_one_prefix(grid, d):
    n, m = len(grid), len(grid[0]); d1 = isqrt(d * d - 1)
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d); B = [(W[c] - A[c]) % MOD if grid[r][c] == '.' else 0 for c in range(m)]
        if r == 0: return (sum(A) + sum(B)) % MOD
        pre = [0] * (m + 1)
        for c in range(m): pre[c + 1] = pre[c] + A[c] + B[c]
        A = [(pre[min(m, c + d1 + 1)] - pre[max(0, c - d1)]) % MOD if grid[r - 1][c] == '.' else 0 for c in range(m)]

rng = random.Random(4)
for _ in range(200):
    n, m = rng.randint(1, 6), rng.randint(1, 6)
    g = ["".join(rng.choice("..#") for _ in range(m)) for _ in range(n)]; d = rng.randint(1, 6)
    assert dp_one_prefix(g, d) == Solution().numberOfRoutes(g, d)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: tối đa hai nước ngang liên tiếp
def two_horizontal_sol(grid, d):
    n, m = len(grid), len(grid[0]); d1 = isqrt(d * d - 1)
    A = [1 if ch == '.' else 0 for ch in grid[n - 1]]
    for r in range(n - 1, -1, -1):
        W = window_sums(A, d); B1 = [(W[c] - A[c]) if grid[r][c] == '.' else 0 for c in range(m)]
        W2 = window_sums(B1, d); B2 = [(W2[c] - B1[c]) if grid[r][c] == '.' else 0 for c in range(m)]
        if r == 0: return (sum(A) + sum(B1) + sum(B2)) % MOD
        U = window_sums([a + b + c2 for a, b, c2 in zip(A, B1, B2)], d1)
        A = [U[c] if grid[r - 1][c] == '.' else 0 for c in range(m)]

def brute_two(grid, d):
    n, m = len(grid), len(grid[0])
    def dfs(r, c, run):
        total = 1 if r == 0 else 0
        for nr in (r, r - 1):
            if nr < 0 or (nr == r and run == 2): continue
            for nc in range(m):
                if grid[nr][nc] != '.' or (nr == r and nc == c): continue
                if (r - nr) ** 2 + (c - nc) ** 2 <= d * d:
                    total += dfs(nr, nc, run + 1 if nr == r else 0)
        return total
    return sum(dfs(n - 1, c, 0) for c in range(m) if grid[n - 1][c] == '.') % MOD

rng = random.Random(5)
for _ in range(200):
    n, m = rng.randint(1, 3), rng.randint(1, 4)
    g = ["".join(rng.choice("..#") for _ in range(m)) for _ in range(n)]; d = rng.randint(1, 3)
    assert two_horizontal_sol(g, d) == brute_two(g, d), (g, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
d1 = isqrt(d*d - 1)
A = [1 if ô mở else 0 cho hàng n-1]
for r = n-1 .. 0:
    B[c] = window(A, d)[c] - A[c]      (ô mở)       # đi ngang tới ô khác
    if r == 0: return sum(A) + sum(B)
    A[c] = window(A + B, d1)[c]        (ô mở hàng r-1)   # đi lên
```

**Checklist:**

- [ ] Bán kính đi lên là `⌊√(d² − 1)⌋`, không phải `d`?
- [ ] Đi ngang tới **ô khác** (trừ `A[c]`)?
- [ ] Đi ngang chỉ từ trạng thái `A` (không hai lần liên tiếp)?
- [ ] Ô bị chặn có giá trị 0 ở cả hai trạng thái?
- [ ] Đáp án gồm cả `A` và `B` của hàng 0; lưới một hàng xử lý đúng?